# AttaCut to ONNX

This notebook demonstrates how to convert the [AttaCut](https://github.com/PyThaiNLP/attacut) Thai word tokenization model from PyTorch to the ONNX format used by LEKCut.

AttaCut provides two model variants:
- **attacut-sc**: Syllable + Character model (recommended, higher accuracy)
- **attacut-c**: Character-only model (simpler, no syllable tokenizer required)


## Prerequisites

Install the required packages:




In [ ]:
import torch
import torch.nn as nn
import onnx
import onnxruntime as ort
import numpy as np
from attacut import Tokenizer


## ONNX Export Wrappers

AttaCut models take a tuple  as input.  We wrap each model
so that ONNX export only sees a single tensor input .
The  argument is not used inside the CNN-based forward passes,
so it is safe to remove.


In [ ]:
class AttacutSCWrapper(nn.Module):
    """Wrapper for attacut-sc (syllable + character) model."""
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        # x shape: (1, 2, seq_len)
        seq_len = torch.tensor([x.shape[2]])
        return self.model((x, seq_len))


class AttacutCWrapper(nn.Module):
    """Wrapper for attacut-c (character-only) model."""
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        # x shape: (1, seq_len)
        seq_len = torch.tensor([x.shape[1]])
        return self.model((x, seq_len))


## Convert attacut-sc


In [ ]:
tok_sc = Tokenizer("attacut-sc")
model_sc = tok_sc.model
model_sc.eval()

wrapper_sc = AttacutSCWrapper(model_sc)
wrapper_sc.eval()

# Dummy input: (batch=1, channels=2, seq_len=10)
dummy_x_sc = torch.zeros((1, 2, 10), dtype=torch.long)

torch.onnx.export(
    wrapper_sc,
    dummy_x_sc,
    "attacut-sc.onnx",
    export_params=True,
    opset_version=18,
    do_constant_folding=True,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={
        "input": {2: "seq_len"},
        "output": {0: "seq_len"},
    },
)
print("Saved attacut-sc.onnx")


## Convert attacut-c


In [ ]:
tok_c = Tokenizer("attacut-c")
model_c = tok_c.model
model_c.eval()

wrapper_c = AttacutCWrapper(model_c)
wrapper_c.eval()

# Dummy input: (batch=1, seq_len=10)
dummy_x_c = torch.zeros((1, 10), dtype=torch.long)

torch.onnx.export(
    wrapper_c,
    dummy_x_c,
    "attacut-c.onnx",
    export_params=True,
    opset_version=18,
    do_constant_folding=True,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={
        "input": {1: "seq_len"},
        "output": {0: "seq_len"},
    },
)
print("Saved attacut-c.onnx")


## Verify the Exported Models

Run the converted models with ONNX Runtime and compare against the
original PyTorch outputs.


In [ ]:
import string, re
import ssg

def character2ix(ch2ix, character):
    if character == "":
        return ch2ix["<PAD>"]
    elif character in string.punctuation:
        return ch2ix.get("<PUNC>", ch2ix["<UNK>"])
    return ch2ix.get(character, ch2ix["<UNK>"])


def syllable2ix(sy2ix, syllable):
    if re.match(r"[A-Za-z]+", syllable):
        token = "<ENGLISH>"
    elif re.match(r"[0-9,]+", syllable):
        token = "<NUMBER>"
    else:
        token = syllable
    return sy2ix.get(token, sy2ix["<UNK>"])


def sigmoid(x):
    x64 = np.asarray(x, dtype=np.float64)
    return 1.0 / (1.0 + np.exp(-np.clip(x64, -500.0, 500.0)))


def find_words_from_preds(tokens, preds):
    curr_word = tokens[0]
    words = []
    for char, pred in zip(tokens[1:], preds[1:]):
        if pred == 0:
            curr_word += char
        else:
            words.append(curr_word)
            curr_word = char
    words.append(curr_word)
    return words


# --- attacut-sc ---
sess_sc = ort.InferenceSession("attacut-sc.onnx")
dataset_sc = tok_sc.dataset
test_text = "ทดสอบการตัดคำ"
tokens, features = dataset_sc.make_feature(test_text)
x_sc, _ = features
logits_sc = sess_sc.run(None, {"input": x_sc.numpy()})[0]
preds_sc = (sigmoid(logits_sc) > 0.5).astype(int)
words_sc = find_words_from_preds(tokens, preds_sc)
print("attacut-sc:", words_sc)

# --- attacut-c ---
sess_c = ort.InferenceSession("attacut-c.onnx")
dataset_c = tok_c.dataset
tokens_c, features_c = dataset_c.make_feature(test_text)
x_c, _ = features_c
logits_c = sess_c.run(None, {"input": x_c.numpy()})[0]
preds_c = (sigmoid(logits_c) > 0.5).astype(int)
words_c = find_words_from_preds(tokens_c, preds_c)
print("attacut-c:", words_c)
